# MI Therapy Chatbot — Fine-Tuning with QLoRA
**Model:** Gemma 2 2B IT | **Method:** QLoRA (4-bit) | **Runtime:** T4 GPU (free)

### How to use this notebook
1. **First time:** Run all cells top to bottom. Enter API keys when prompted — they get saved to Google Drive so you never enter them again.
2. **Next time:** Just "Runtime → Run all" — everything auto-loads.
3. **Watch training live:** Click the wandb link that appears during training.

## Step 1 — Install Libraries & Mount Google Drive

In [ ]:
# ===== STEP 1a: Fix numpy/scipy compatibility (run FIRST, then restart runtime) =====
# Colab's latest numpy 2.x breaks scipy — downgrade to compatible set
!pip uninstall -y numpy scipy scikit-learn
!pip install -q numpy==1.26.4 scipy==1.13.1 scikit-learn==1.5.2

# Verify install
import importlib, numpy; importlib.reload(numpy)
print(f'numpy version: {numpy.__version__}')

print('\n⚠️  NOW RESTART THE RUNTIME: Runtime → Restart runtime')
print('   Then SKIP this cell and run Step 1b below')

In [ ]:
# ===== STEP 1b: Install ML libraries & mount Drive (run AFTER restart) =====
!pip install -q transformers>=4.44.0 peft>=0.12.0 trl>=0.10.0 accelerate>=0.33.0
!pip install -q datasets wandb huggingface_hub
!pip install -q bitsandbytes>=0.46.1

# ===== MOUNT GOOGLE DRIVE =====
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/mi-therapy-capstone'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/adapters', exist_ok=True)

print(f'\n✅ Libraries installed')
print(f'✅ Google Drive mounted at: {PROJECT_DIR}')
print(f'\nIMPORTANT: Upload your data files to:')
print(f'   Google Drive → My Drive → mi-therapy-capstone → data/')
print(f'   - finetune_train.jsonl')
print(f'   - finetune_eval.jsonl')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    raise RuntimeError('❌ No GPU! Go to Runtime → Change runtime type → T4 GPU')

## Step 2 — API Keys (saved to Google Drive, enter only once ever)

In [ ]:
import json, os

KEYS_FILE = f'{PROJECT_DIR}/api_keys.json'

# Load saved keys or prompt for new ones
if os.path.exists(KEYS_FILE):
    with open(KEYS_FILE) as f:
        keys = json.load(f)
    print('✅ Loaded saved API keys from Google Drive')
else:
    print('First time setup — enter your API keys (you only do this once):\n')
    keys = {}
    keys['hf_token'] = input('Paste your HuggingFace token (from hf.co/settings/tokens): ').strip()
    keys['wandb_key'] = input('Paste your Wandb API key (from wandb.ai/settings): ').strip()
    keys['hf_username'] = input('Your HuggingFace username: ').strip()

    with open(KEYS_FILE, 'w') as f:
        json.dump(keys, f)
    print(f'\n✅ Keys saved to Google Drive — you will never need to enter them again')

# Login to both services
from huggingface_hub import login
login(token=keys['hf_token'], add_to_git_credential=False)
print(f'✅ Logged into HuggingFace as: {keys["hf_username"]}')

import wandb
wandb.login(key=keys['wandb_key'], relogin=True)
print(f'✅ Logged into Weights & Biases')

HF_USERNAME = keys['hf_username']

## Step 3 — Load Training Data (from Google Drive)

**Before running this cell:** Upload your 2 data files to Google Drive:

Google Drive → My Drive → `mi-therapy-capstone` → `data` → upload:
- `finetune_train.jsonl`
- `finetune_eval.jsonl`

(These files are at `C:\Users\tirth\capstone-chatbot\data\processed\` on your PC)

In [ ]:
import json
from datasets import Dataset

DATA_DIR = f'{PROJECT_DIR}/data'

def load_jsonl(path):
    data = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

# Check files exist
train_path = f'{DATA_DIR}/finetune_train.jsonl'
eval_path  = f'{DATA_DIR}/finetune_eval.jsonl'

assert os.path.exists(train_path), f'❌ File not found: {train_path}\nUpload finetune_train.jsonl to Google Drive → mi-therapy-capstone → data'
assert os.path.exists(eval_path),  f'❌ File not found: {eval_path}\nUpload finetune_eval.jsonl to Google Drive → mi-therapy-capstone → data'

train_data = load_jsonl(train_path)
eval_data  = load_jsonl(eval_path)

train_dataset = Dataset.from_list(train_data)
eval_dataset  = Dataset.from_list(eval_data)

# Fix: Gemma 2 doesn't support system role — merge into first user message
def fix_messages(example):
    messages = example['messages']
    fixed = []
    system_text = ''
    for msg in messages:
        if msg['role'] == 'system':
            system_text += msg['content'] + '\n'
        else:
            if system_text and msg['role'] == 'user' and not fixed:
                msg = {'role': 'user', 'content': system_text + msg['content']}
                system_text = ''
            fixed.append(msg)
    if system_text:
        fixed.insert(0, {'role': 'user', 'content': system_text.strip()})
    example['messages'] = fixed
    return example

train_dataset = train_dataset.map(fix_messages)
eval_dataset = eval_dataset.map(fix_messages)

# Verify no system roles remain
for split_name, ds in [('train', train_dataset), ('eval', eval_dataset)]:
    roles = set(r for ex in ds for r in [m['role'] for m in ex['messages']])
    print(f'  {split_name} roles: {roles}')

print(f'\n✅ Train: {len(train_data)} conversations')
print(f'✅ Eval:  {len(eval_data)} conversations')
print(f'   Avg messages/convo: {sum(len(d["messages"]) for d in train_data) / len(train_data):.1f}')

## Step 4 — Load Gemma 2 2B in 4-bit (QLoRA)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_ID = 'google/gemma-2-2b-it'

# 4-bit quantization config (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

print('Loading model in 4-bit (this takes 1-2 minutes)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)

print(f'\n✅ Model loaded!')
print(f'   Parameters: {model.num_parameters()/1e9:.2f}B')
print(f'   GPU memory used: {torch.cuda.memory_allocated()/1024**3:.2f} GB')

## Step 5 — Configure LoRA Adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA adapters attached!')
print(f'   Trainable: {trainable:,} params ({100*trainable/total:.2f}% of {total:,})')
print(f'   Frozen:    {total - trainable:,} params')

## Step 6 — Train!

In [ ]:
from trl import SFTTrainer, SFTConfig
import time

# Version your runs: v1, v2, v3... change this each time you retrain
VERSION = 'v2'
RUN_NAME = f'mi-therapy-gemma2-{VERSION}'
ADAPTER_DIR = f'{PROJECT_DIR}/adapters/{RUN_NAME}'

training_args = SFTConfig(
    output_dir=ADAPTER_DIR,              # save checkpoints directly to Google Drive

    run_name=RUN_NAME,

    # Training schedule
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_ratio=0.1,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',

    # Memory optimization (bf16 required for 4-bit quantized model)
    fp16=False,
    bf16=True,
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,

    # Eval disabled to save VRAM on T4
    eval_strategy='no',
    save_steps=12,                       # checkpoint every 12 steps to Drive
    load_best_model_at_end=False,

    # Logging to wandb
    report_to='wandb',
    logging_steps=5,

    # Misc
    max_grad_norm=0.3,
    group_by_length=True,
    dataloader_pin_memory=False,

    # SFT-specific (trl 1.0.0)
    max_length=2048,
    packing=False,
)

wandb.init(project='mi-therapy-chatbot', name=RUN_NAME)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,          # trl 1.0.0: use processing_class not tokenizer
)

steps_per_epoch = len(train_data) // (1 * 8)
total_steps = steps_per_epoch * 3
print(f'✅ Training config ready!')
print(f'   ~{steps_per_epoch} steps/epoch × 3 epochs = ~{total_steps} total steps')
print(f'   Checkpoints save to Google Drive every 12 steps')
print(f'   Adapter version: {VERSION}')
print(f'\n🚀 Click the wandb link above to watch training LIVE')
print(f'   Look for: train_loss going DOWN\n')

trainer.train()

# ===== AUTO-SAVE immediately after training (same cell = can't be missed) =====
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'\n✅ Adapter auto-saved to Google Drive: {ADAPTER_DIR}')

## Step 7 — Save Adapter (to Google Drive + Hugging Face)

In [ ]:
# Adapter already saved to Google Drive by Step 6
# This cell just pushes to HuggingFace as a backup

repo_name = f'{HF_USERNAME}/{RUN_NAME}'
try:
    model.push_to_hub(repo_name, private=True)
    tokenizer.push_to_hub(repo_name, private=True)
    print(f'✅ Pushed to: huggingface.co/{repo_name}')
except Exception as e:
    print(f'⚠️  HuggingFace push failed: {e}')
    print(f'   No worries — adapter is safe on Google Drive: {ADAPTER_DIR}')
    print(f'   To fix: create a Write token at huggingface.co/settings/tokens')

## Step 8 — Quick Test

In [ ]:
from transformers import pipeline

pipe = pipeline('text-generation', model=model, tokenizer=tokenizer,
                max_new_tokens=200, temperature=0.7, do_sample=True)

# Test 1: High defensiveness client
test1 = [
    {"role": "user", "content": "[Session begins. Client enters.]"},
    {"role": "assistant", "content": "Thanks for coming in today. What brings you here?"},
    {"role": "user", "content": "I don't even want to be here. My wife made me come. I don't have a drinking problem.\n[emotion:angry, defensiveness:high]"},
]

# Test 2: Sad, low defensiveness
test2 = [
    {"role": "user", "content": "[Session begins. Client enters.]"},
    {"role": "assistant", "content": "It's good to see you again. How has your week been?"},
    {"role": "user", "content": "Not great. I drank every night again. I'm just... I don't know why I keep doing this.\n[emotion:sad, defensiveness:low]"},
]

print('=' * 60)
print('TEST 1: Angry, high defensiveness')
print('=' * 60)
r1 = pipe(test1)
print(r1[0]['generated_text'][-1]['content'])

print('\n' + '=' * 60)
print('TEST 2: Sad, low defensiveness')
print('=' * 60)
r2 = pipe(test2)
print(r2[0]['generated_text'][-1]['content'])

try:
    wandb.finish()
except:
    pass
print('\n✅ All done! Check wandb for the full training report.')